In [11]:
import os
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

HAIKU = "claude-haiku-4-5-20251001"
SONNET = "claude-sonnet-4-6"
DEFAULT_MODEL = HAIKU
load_dotenv()

def get_model(model_name: str = DEFAULT_MODEL, temperature: float = 0.0, max_tokens: int = 2048) -> ChatAnthropic:
    """Renvoie une instance configurée de ChatAnthropic.
    Lève une erreur claire si la clé API n'est pas trouvée.
    """
    if not os.getenv("ANTHROPIC_API_KEY"):
        raise RuntimeError(
            "ANTHROPIC_API_KEY introuvable. "
            "As-tu créé un fichier .env à partir de .env ?"
        )
    return ChatAnthropic(model=model_name, temperature=temperature, max_tokens=max_tokens)


In [30]:

# 1. Définir le gabarit de prompt
prompt = ChatPromptTemplate.from_template("Raconte-moi une blague courte sur le thème : {sujet}")
# 2. Initialiser le modèle (nécessite la clé API)
model = get_model()
# 3. Créer le parser de sortie
parser = StrOutputParser()
# 4. Assembler la chaîne (LCEL)
chain = prompt | model | parser
# 5. Exécuter la chaîne
reponse = chain.invoke({"sujet": "Ibrahim ALAME"})
print(reponse)

# La blague d'Ibrahim Alame 😄

Ibrahim Alame rentre chez lui et trouve sa femme en train de crier :
— "Ibrahim ! Il y a un cambrioleur dans la maison !"

Ibrahim répond calmement :
— "Pas de problème, je vais lui faire une blague... Il va rire tellement qu'il va oublier pourquoi il est venu !" 

*Et c'est comme ça qu'Ibrahim a sauvé sa maison avec son humour légendaire !* 🎤

---

*(Ibrahim Alame est un humoriste connu pour son style comique très particulier et ses blagues absurdes)*


<H2><font color="#b22222"> Schéma de sortie structurée </font> </H2>

On force le LLM à répondre dans ce schéma exact.

Avantage énorme par rapport à parser du markdown ou du JSON à la main :
   - Jamais de malformation (le SDK Anthropic utilise tool-calling sous le capot)
   - Tu obtiens un objet Python typé, auto-complété par PyCharm
   - Les `description=...` sont envoyés au modèle comme instructions


In [13]:
from pydantic import BaseModel, Field

class ComprehensionQuestion(BaseModel):
    """Une question de compréhension pour vérifier l'acquisition."""

    question: str = Field(description="La question posée à l'élève.")
    expected_answer: str = Field(
        description="La réponse attendue, en une à deux phrases."
    )

class Explanation(BaseModel):
    """Réponse pédagogique complète sur un concept Python."""

    concept: str = Field(description="Le nom du concept expliqué.")
    intuition: str = Field(
        description=(
            "Explication intuitive en exactement 3 phrases. "
            "Évite le jargon, vise la clarté pour l'élève."
        )
    )
    code_example: str = Field(
        description=(
            "Un exemple de code Python minimal et commenté qui illustre "
            "le concept. Préfère des noms de variables parlants."
        )
    )
    pitfall: str = Field(
        description="Un piège classique que rencontrent les élèves sur ce concept."
    )
    comprehension_questions: list[ComprehensionQuestion] = Field(
        description="Exactement 3 questions de compréhension, de difficulté croissante."
    )



<H2><font color="#b22222">Le prompt template</font></H2>

`{level}` et `{concept}` sont des variables qui seront remplacées au
moment de l'invoke. Le prompt est lui-même un Runnable composable.


In [14]:
EXPLAIN_PROMPT = ChatPromptTemplate.from_messages([
    (
        "system",
        "Tu es un assistant pédagogique pour des élèves apprenant Python. "
        "Tu adaptes systématiquement ton vocabulaire et tes exemples au "
        "niveau de l'élève. Niveau actuel : {level}.\n\n"
        "Règles selon le niveau :\n"
        "  - 'débutant' : pas de jargon, exemples ultra-simples, "
        "analogies du quotidien, pas plus de 10 lignes de code.\n"
        "  - 'intermédiaire' : tu peux supposer la syntaxe de base "
        "connue, montre les usages idiomatiques (PEP 8, type hints).\n"
        "  - 'avancé' : montre les détails du modèle objet de Python, "
        "les subtilités, les cas limites, les implications de performance."
    ),
    ("human", "Explique le concept suivant : {concept}"),
])


<H2><font color="#b22222"> La chaîne LCEL </font> </H2>

<pre>`prompt | model | (rien après, with_structured_output remplace le parser)`</pre>

`with_structured_output(Explanation)` configure le modèle pour qu'il
remplisse le schéma Pydantic. Tu peux l'imaginer comme un parser intégré
au modèle, plus fiable qu'un PydanticOutputParser à la sortie (que tu
verras dans des vieux tutos — il marche, mais c'est moins propre).

In [15]:
def explain_chain_for_level(level: str):
    """Renvoie une chaîne dont le niveau est figé.
    `.partial(level=...)` crée un nouveau prompt avec une variable fixée.
    On peut donc construire des chaînes spécialisées sans dupliquer le template.
    """
    model = get_model().with_structured_output(Explanation)
    return EXPLAIN_PROMPT.partial(level=level) | model

<H2><font color="#b22222"> Génération multi-niveaux en parallèle </font> </H2>

RunnableParallel exécute ses sous-chaînes EN CONCURRENCE et renvoie
un dict avec les résultats. Les deux appels au LLM partent en même temps :
tu paies la latence d'UN seul appel au lieu de deux.
Ouvre <b>LangSmith</b> après l'exécution, tu verras visuellement les deux
traces lancées simultanément.

In [16]:
from langchain_core.runnables import RunnableParallel


def build_multi_level_chain():
    """Génère les versions débutant ET intermédiaire en un seul invoke."""
    return RunnableParallel(
        beginner=explain_chain_for_level("débutant"),
        intermediate=explain_chain_for_level("intermédiaire"),
    )

<H2><font color="#b22222"> Variante streaming </font> </H2>

`with_structured_output` ne streame pas bien (le modèle doit produire
l'objet complet pour qu'il soit valide). Pour du streaming token par
token, on utilise une chaîne classique avec sortie texte.

In [17]:
STREAM_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "Tu es un assistant pédagogique Python concis et précis."),
    ("human", "Explique en 2 paragraphes : {concept}"),
])


def build_streaming_chain():
    """Chaîne texte simple, idéale pour du streaming dans un terminal/UI."""
    return STREAM_PROMPT | get_model() | StrOutputParser()

<H2><font color="#b22222"> Démo </font></H2>

In [18]:
def demo() -> None:
    concept = "les générateurs Python"

    # --- Démo 1 : sortie structurée mono-niveau ---
    print("=" * 70)
    print(f"DÉMO 1 — Explication structurée : '{concept}' (intermédiaire)")
    print("=" * 70)
    chain = explain_chain_for_level("intermédiaire")
    result: Explanation | any = chain.invoke({"concept": concept})

    # `result` est un vrai objet Python : tu peux faire result.intuition,
    # PyCharm autocomplète, mypy est content.
    print(f"\nIntuition :\n{result.intuition}\n")
    print(f"Code :\n{result.code_example}\n")
    print(f"Piège classique :\n{result.pitfall}\n")
    print("Questions de compréhension :")
    for i, q in enumerate(result.comprehension_questions, 1):
        print(f"  {i}. {q.question}")
        print(f"     → {q.expected_answer}")

    # --- Démo 2 : multi-niveaux en parallèle ---
    print("\n" + "=" * 70)
    print(f"DÉMO 2 — RunnableParallel : 2 niveaux en concurrence")
    print("=" * 70)
    multi = build_multi_level_chain()
    results = multi.invoke({"concept": concept})

    print("\n--- Pour un DÉBUTANT ---")
    print(results["beginner"].intuition)
    print("\n--- Pour un INTERMÉDIAIRE ---")
    print(results["intermediate"].intuition)

    # --- Démo 3 : streaming ---
    print("\n" + "=" * 70)
    print("DÉMO 3 — Streaming token par token")
    print("=" * 70)
    print()
    streamer = build_streaming_chain()
    for chunk in streamer.stream({"concept": "le GIL en Python"}):
        print(chunk, end="", flush=True)
    print("\n")


if __name__ == "__main__":
    demo()



DÉMO 1 — Explication structurée : 'les générateurs Python' (intermédiaire)

Intuition :
Un générateur est une fonction qui produit une séquence de valeurs une à la fois, au lieu de tout calculer et retourner d'un coup. Imagine une usine qui fabrique des objets à la demande plutôt que de les stocker tous en entrepôt : elle économise de l'espace et de l'énergie. En Python, les générateurs utilisent le mot-clé `yield` pour "pausable" et reprendre l'exécution, ce qui les rend très efficaces en mémoire.

Code :

# Fonction classique : crée une liste complète en mémoire
def nombres_classique(n):
    resultat = []
    for i in range(n):
        resultat.append(i ** 2)
    return resultat

# Générateur : produit les valeurs une à une
def nombres_generateur(n):
    for i in range(n):
        yield i ** 2

# Utilisation
print("Classique :", nombres_classique(5))  # [0, 1, 4, 9, 16]

gen = nombres_generateur(5)
print("Générateur :", gen)  # <generator object nombres_generateur at ...>
print("Vale


<H3> <font color="6495ed"> EXERCICE 1 — Multi-niveaux 3x : parallèle vs séquentiel </font></H3>

Objectif : montrer concrètement que RunnableParallel ne paye PAS la somme
des latences mais le MAX. Avec 3 appels qui prennent ~2s chacun, le séquentiel
tombe à ~6s, le parallèle à ~2s.


In [19]:
import time

print("\n" + "=" * 70)
print("EXERCICE 1 — Parallèle vs séquentiel sur 3 niveaux")
print("=" * 70)

concept = "les décorateurs Python"

# --- Construction des 3 chaînes spécialisées ---
beginner = explain_chain_for_level("débutant")
intermediate = explain_chain_for_level("intermédiaire")
advanced = explain_chain_for_level("avancé")

# --- Version SÉQUENTIELLE : 3 invokes successifs ---
t0 = time.perf_counter()
seq_results = {
    "beginner": beginner.invoke({"concept": concept}),
    "intermediate": intermediate.invoke({"concept": concept}),
    "advanced": advanced.invoke({"concept": concept}),
}
seq_time = time.perf_counter() - t0
print(f"\nSéquentiel : {seq_time:.2f}s pour 3 niveaux")

# --- Version PARALLÈLE : un seul invoke sur un RunnableParallel ---
parallel = RunnableParallel(
    beginner=beginner,
    intermediate=intermediate,
    advanced=advanced,
)
t0 = time.perf_counter()
par_results = parallel.invoke({"concept": concept})
par_time = time.perf_counter() - t0
print(f"Parallèle  : {par_time:.2f}s pour 3 niveaux")
print(f"→ Speedup  : ×{seq_time / par_time:.2f}")

# On affiche l'intuition avancée pour vérifier que ça a bien tourné
print(f"\nExtrait niveau avancé :\n{par_results['advanced'].intuition[:200]}...")



EXERCICE 1 — Parallèle vs séquentiel sur 3 niveaux

Séquentiel : 23.21s pour 3 niveaux
Parallèle  : 9.89s pour 3 niveaux
→ Speedup  : ×2.35

Extrait niveau avancé :
Un décorateur est une fonction qui prend une autre fonction en entrée, la modifie ou l'enveloppe, puis retourne la version modifiée. C'est comme un emballage cadeau : tu prends un objet (ta fonction),...


<H3> <font color="6495ed"> EXERCICE 2 — batch() vs boucle for + invoke </font></H3>

#### Objectif :
montrer que .batch() paralléliser plusieurs INPUTS sur la MÊME
chaîne. À ne pas confondre avec `RunnableParallel` qui parallélise plusieurs
CHAÎNES sur le même input.

#### Règle simple :
  - plusieurs inputs, même chaîne   `-> .batch(inputs)`
  - même input, plusieurs chaînes   `-> RunnableParallel(a=c1, b=c2, ...)`

In [20]:
print("\n" + "=" * 70)
print("EXERCICE 2 — batch() vs boucle for")
print("=" * 70)

concepts = ["les décorateurs", "les métaclasses", "les générateurs", "le GIL"]
chain = explain_chain_for_level("intermédiaire")

# --- Version BOUCLE : un invoke après l'autre ---
t0 = time.perf_counter()
loop_results = [chain.invoke({"concept": c}) for c in concepts]
loop_time = time.perf_counter() - t0
print(f"\nBoucle for : {loop_time:.2f}s pour {len(concepts)} concepts")

# --- Version BATCH : un seul appel, parallélisé sous le capot ---
inputs = [{"concept": c} for c in concepts]
t0 = time.perf_counter()
batch_results = chain.batch(inputs)
batch_time = time.perf_counter() - t0
print(f"batch()    : {batch_time:.2f}s pour {len(concepts)} concepts")
print(f"→ Speedup  : ×{loop_time / batch_time:.2f}")

# Note : tu peux contrôler le degré de parallélisme avec max_concurrency
# pour éviter de saturer les rate limits :
#   chain.batch(inputs, config={"max_concurrency": 5})

print(f"\nExemple ({concepts[0]}) :\n{batch_results[0].intuition[:200]}...")


EXERCICE 2 — batch() vs boucle for

Boucle for : 30.04s pour 4 concepts
batch()    : 9.33s pour 4 concepts
→ Speedup  : ×3.22

Exemple (les décorateurs) :
Un décorateur est une fonction qui "enveloppe" une autre fonction pour modifier son comportement sans changer son code original. C'est comme ajouter une couche de papier cadeau autour d'un cadeau : le...


<H3> <font color="6495ed"> EXERCICE 3 — Schéma Pydantic enrichi avec un champ "analogy" </font></H3>

#### Objectif :
    montrer la trivialité d'ajouter un champ. La seule chose à faire, c'est étendre le schéma. Le modèle remplit automatiquement le nouveau champ grâce au mécanisme de `tool-calling` sous-jacent.

C'est LA force de `with_structured_output` : tu modifies juste ta classe `Pydantic`, et le contrat avec le LLM se met à jour automatiquement.

In [21]:
class ExplanationWithAnalogy(BaseModel):
    """Version enrichie : on rajoute une analogie du quotidien."""

    concept: str = Field(description="Le concept expliqué.")
    intuition: str = Field(description="Explication intuitive en 3 phrases.")
    analogy: str = Field(
        description=(
            "Une analogie avec un objet ou une situation du quotidien "
            "(cuisine, sport, bureau...) qui éclaire le concept pour un débutant. "
            "Une seule phrase percutante."
        )
    )
    code_example: str = Field(description="Un exemple de code minimal et commenté.")
    pitfall: str = Field(description="Un piège classique sur ce concept.")

In [22]:
print("\n" + "=" * 70)
print("EXERCICE 3 — Schéma Pydantic enrichi avec un champ analogy")
print("=" * 70)

# Une seule ligne change par rapport à la Phase 1 :
# le schéma passé à with_structured_output.
model = get_model().with_structured_output(ExplanationWithAnalogy)
chain = EXPLAIN_PROMPT.partial(level="débutant") | model

for concept in ["les décorateurs", "le polymorphisme", "le contexte avec `with`"]:
    result = chain.invoke({"concept": concept})
    print(f"\n--- {concept} ---")
    print(f"Analogie : {result.analogy}")


EXERCICE 3 — Schéma Pydantic enrichi avec un champ analogy

--- les décorateurs ---
Analogie : C'est comme emballer un cadeau : tu prends ta fonction (le cadeau), tu la mets dans du papier cadeau (le décorateur), et maintenant elle a une belle présentation en plus, mais c'est toujours le même cadeau à l'intérieur.

--- le polymorphisme ---
Analogie : C'est comme une télécommande universelle : tu appuies sur le même bouton "volume", mais il augmente le son de la TV, de la radio, ou du téléphone selon l'appareil — le bouton est identique, l'effet change.

--- le contexte avec `with` ---
Analogie : C'est comme emprunter un livre à la bibliothèque : `with` c'est le système qui te le donne et qui le reprend automatiquement à la date limite, sans que tu aies besoin de te souvenir de le rendre.


<H3> <font color="6495ed"> EXERCICE 4 — Router avec RunnableBranch et classifieur LLM </font></H3>

#### Objectif :
faire dispatcher dynamiquement vers 3 sous-chaînes selon le type
de concept. Pattern fondamental, qu'on retrouvera partout en production.

#### Architecture :
<pre>
   {"concept": "lambda"}
         |
         |  RunnablePassthrough.assign(category=classifier_chain)
         v
   {"concept": "lambda", "category": "syntaxe"}
         |
         |  RunnableBranch (3 conditions + un défaut)
         v
   réponse spécialisée syntaxe
</pre>

In [26]:
from typing import Literal
from langchain_core.runnables import RunnablePassthrough, RunnableBranch, RunnableLambda


class CategoryResult(BaseModel):
    """Catégorie d'un concept Python pour router vers la bonne sous-chaîne."""

    category: Literal["syntaxe", "paradigme", "ecosysteme"] = Field(
        description=(
            "Catégorise le concept :\n"
            " - 'syntaxe' : éléments du langage (def, lambda, comprehensions, with, yield...)\n"
            " - 'paradigme' : concepts de design (POO, fonctionnel, héritage, polymorphisme...)\n"
            " - 'ecosysteme' : outils/bibliothèques (pip, venv, pytest, numpy, requests...)"
        )
    )


def build_classifier():
    """Mini-chaîne qui catégorise un concept en 3 classes."""
    classify_prompt = ChatPromptTemplate.from_messages([
        ("system", "Tu classes des concepts Python en 3 catégories. Ne réponds que par la catégorie."),
        ("human", "Concept à catégoriser : {concept}"),
    ])
    # Important : on récupère .category (le string) plutôt que l'objet entier
    # pour pouvoir l'utiliser facilement dans les prédicats de RunnableBranch.
    return classify_prompt | get_model().with_structured_output(CategoryResult) | RunnableLambda(lambda r: r.category)


def build_specialized_chain(focus: str):
    """Construit une sous-chaîne avec un prompt système orienté `focus`."""
    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "Tu es un assistant Python. Le concept à expliquer relève de la catégorie "
         f"'{focus}'. Adapte ton explication en conséquence :\n" +
         {
             "syntaxe":   "insiste sur la syntaxe exacte, montre un mini exemple et un anti-exemple.",
             "paradigme": "insiste sur la philosophie, le 'pourquoi', les implications de design.",
             "ecosysteme": "insiste sur l'usage pratique, l'installation, les commandes shell typiques.",
         }[focus]),
        ("human", "Explique : {concept}"),
    ])
    return prompt | get_model()


def build_router():
    """Le routeur complet : classifie puis branche."""
    classifier = build_classifier()
    syntaxe_chain    = build_specialized_chain("syntaxe")
    paradigme_chain  = build_specialized_chain("paradigme")
    ecosysteme_chain = build_specialized_chain("ecosysteme")

    # RunnableBranch prend N couples (prédicat, chaîne) + une chaîne par défaut.
    # Le prédicat reçoit le dict courant et renvoie True/False.
    branch = RunnableBranch(
        (lambda x: x["category"] == "syntaxe",    syntaxe_chain),
        (lambda x: x["category"] == "paradigme",  paradigme_chain),
        ecosysteme_chain,  # défaut : si rien ne matche
    )

    # On enrichit le dict d'entrée avec la catégorie, puis on branche.
    # Note : .assign() conserve la clé "concept" pour que les sous-chaînes
    # puissent toujours lire {concept} dans leur template.
    return RunnablePassthrough.assign(category=classifier) | branch



In [27]:
print("\n" + "=" * 70)
print("EXERCICE 4 — Router avec RunnableBranch")
print("=" * 70)

router = build_router()

test_concepts = [
    "les list comprehensions",   # syntaxe attendue
    "le polymorphisme",           # paradigme attendu
    "pytest",                     # écosystème attendu
    "lambda",                     # syntaxe
]

# On veut aussi voir la catégorie choisie. On reconstruit le pipeline en
# gardant la catégorie dans la sortie pour la démo (en prod on s'en passe).
classifier = build_classifier()
debug_chain = RunnablePassthrough.assign(
    category=classifier,
    answer=router,
)

for concept in test_concepts:
    result = debug_chain.invoke({"concept": concept})
    category = result["category"]
    # AIMessage -> on prend les 200 premiers caractères du contenu textuel
    snippet = result["answer"].content[:200].replace("\n", " ")
    print(f"\n[{category:>10}] {concept}")
    print(f"             → {snippet}...")



EXERCICE 4 — Router avec RunnableBranch

[   syntaxe] les list comprehensions
             → # Les List Comprehensions en Python  ## Concept Une **list comprehension** est une syntaxe compacte pour créer une liste en appliquant une opération à chaque élément d'une séquence, souvent avec un fi...

[ paradigme] le polymorphisme
             → # Le Polymorphisme : Une Philosophie de Flexibilité  ## L'Essence Philosophique  Le polymorphisme incarne une idée fondamentale : **"une même interface, plusieurs formes"**. C'est la capacité d'une en...

[ecosysteme] pytest
             → # pytest - Framework de Test Python  ## 🎯 Concept Clé **pytest** est un framework de test moderne et puissant pour Python. Il simplifie l'écriture et l'exécution de tests avec une syntaxe minimaliste ...

[   syntaxe] lambda
             → # Lambda en Python  ## Concept Une **lambda** est une fonction anonyme (sans nom) définie en une seule ligne. Elle permet de créer rapidement une petite fonction sans utiliser 

<H3> <font color="6495ed"> EXERCICE 5 — Async natif avec asyncio.gather </font></H3>

### Objectif :
montrer la voie async pure et comparer avec RunnableParallel.

#### Conclusion attendue :
les deux donnent des temps quasi identiques parce
que RunnableParallel utilise asyncio sous le capot. La différence est
#### stylistique :
   - asyncio.gather : tu écris le contrôle d'exécution explicitement
   - RunnableParallel : tu décris la composition, l'exécution est implicite

En pratique, on utilise RunnableParallel dans les chaînes (composable),
et asyncio.gather dans le code applicatif (boucle de chat, batch jobs...).

In [28]:
import asyncio
print("\n" + "=" * 70)
print("EXERCICE 5 — Async avec asyncio.gather vs RunnableParallel")
print("=" * 70)

concept = "les générateurs Python"
beginner = explain_chain_for_level("débutant")
intermediate = explain_chain_for_level("intermédiaire")
advanced = explain_chain_for_level("avancé")

# --- Version asyncio.gather pure ---
# On utilise .ainvoke (la variante async) sur chaque chaîne, et gather
# attend toutes les coroutines en concurrence.
t0 = time.perf_counter()
beg_r, int_r, adv_r = await asyncio.gather(
    beginner.ainvoke({"concept": concept}),
    intermediate.ainvoke({"concept": concept}),
    advanced.ainvoke({"concept": concept}),
)
gather_time = time.perf_counter() - t0
print(f"\nasyncio.gather    : {gather_time:.2f}s")

# --- Version RunnableParallel avec .ainvoke (async aussi) ---
parallel = RunnableParallel(beginner=beginner, intermediate=intermediate, advanced=advanced)
t0 = time.perf_counter()
results = await parallel.ainvoke({"concept": concept})
parallel_time = time.perf_counter() - t0
print(f"RunnableParallel  : {parallel_time:.2f}s")

# --- Version synchrone pour la référence ---
t0 = time.perf_counter()
parallel.invoke({"concept": concept})
sync_time = time.perf_counter() - t0
print(f"Sync (RunnablePar): {sync_time:.2f}s")

print(f"\nÉchantillon ({concept}, débutant) :")
print(beg_r.intuition[:200] + "...")



EXERCICE 5 — Async avec asyncio.gather vs RunnableParallel

asyncio.gather    : 9.91s
RunnableParallel  : 8.90s
Sync (RunnablePar): 8.09s

Échantillon (les générateurs Python, débutant) :
Un générateur est une fonction spéciale qui produit des valeurs une à la fois, au lieu de tout calculer d'un coup. Imagine une usine qui fabrique des bonbons : au lieu de fabriquer 1000 bonbons et les...


In [29]:
print("\n" + "=" * 70)
print("✓ Tous les exercices de la Phase 1 sont résolus.")
print("=" * 70)


✓ Tous les exercices de la Phase 1 sont résolus.
